앞서 만든 RAG retriever를 활용해 '키키테크'의 사내 정보에 대해 알려주는 QA 에이전트를 만들어 봅시다!

우선 앞의 실습에서 열심히 만든 retriever tool 코드를 가져옵시다!

RAG는 의미 기반으로 문서를 검색하기 때문에, "이서연의 내선번호 알려줘"처럼 정확한 값을 찾는 질문에는 오히려 부정확할 수 있습니다. 

`search_employee`는 엑셀 파일을 pandas로 직접 읽어 이름으로 정확하게 찾아오기 때문에 이런 경우에 훨씬 신뢰할 수 있습니다.

Agent는 질문을 보고 RAG tool과 이 tool 중 어떤 걸 쓸지 스스로 판단합니다.

In [ ]:
import pandas as pd
from langchain_core.tools import tool

@tool
def search_employee(name: str) -> str:
    """임직원 이름으로 정보를 검색합니다. 연락처, 부서, 담당업무 등을 확인할 수 있습니다."""
    df = pd.read_excel("data/키키테크_임직원및프로젝트현황.xlsx", header=1)
    result = df[df["성명"] == name]

    if result.empty:
        return f"'{name}' 이름의 임직원을 찾을 수 없습니다."

    row = result.iloc[0]
    return "\n".join(f"{col}: {row[col]}" for col in df.columns)


#### 가드레일

Agent에게 "키키테크 관련 질문에만 답해"라고 system prompt로 지시할 수도 있지만, 교묘하게 우회하는 질문에는 뚫리기도 합니다.

가드레일은 Agent를 실행하기 전에 LLM으로 먼저 질문을 검사해서, 관련 없는 질문은 아예 Agent까지 도달하지 못하도록 막는 방식입니다.

비용 절감과 보안, 두 가지 효과를 동시에 얻을 수 있습니다.

In [ ]:
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv, find_dotenv

# 프로젝트 루트의 .env 파일을 자동으로 찾아서 불러오기
load_dotenv(find_dotenv())

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

llm = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

In [ ]:
from langchain.agents.middleware import before_agent
from langchain.agents import AgentState
from langchain_core.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

@before_agent(can_jump_to=["end"])
def guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    query = state["messages"][-1].content
    response = .invoke(
        # llm에게 전달할 프롬프트를 작성해 봅시다.
        "???"
    )
    if "true" not in response.content.lower():
        return {
            "messages": [AIMessage("죄송합니다. 키키테크 관련 질문에만 답변드릴 수 있습니다.")],
            "jump_to": "end"
        }
    return None


In [ ]:
agent = create_agent(

)

완성한 에이전트를 테스트해 봅시다